<a href="https://colab.research.google.com/github/alter-mix-dev/BOTVETERINARIO/blob/feat%2Fpruebas/agente_veterinario.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: DISEÑO DE AGENTES CONVERSACIONALES**

Una vez vista la masterclass ***Diseño de Agentes Conversacionales***, se proporciona el siguiente ***colab*** para que **el alumnado pueda practicar** haciendo uso del lenguaje de programación ***python*** y el modelo de ***llama 3.1*** de Meta.

## **LÓGICA DE NEGOCIO Y MODELO**

### **CONFIGURACIÓN DEL ENTORNO**


#### **COLAB SECRETS**

Para no exponer tu API key directamente en el código, Colab ofrece un panel de Secrets (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la API key para guardarla dentro de una variable y usarla dentro del notebook.

In [1]:
# Leer credenciales desde Colab Secrets

from google.colab import userdata

VERIFY_TOKEN = "mi_token"
WHATSAPP_TOKEN = userdata.get('WHATSAPP_ACCESS_TOKEN')
PHONE_ID = userdata.get('PHONE_NUMBER_ID')

print("Credenciales cargadas correctamente.")

Credenciales cargadas correctamente.


#### **INSTALAR Y ARRANCAR OLLAMA EN COLAB**

Este módulo corre en Colab, así que Ollama no viene preinstalado en la máquina virtual: se instala y se arranca en segundo plano al inicio de cada sesión.

In [2]:
# Instalar Ollama

!apt-get update -qq
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh
print ("Ollama listo")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 64 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
# Iniciar el servidor de Ollama en segundo plano

import subprocess
import time

proceso_ollama = subprocess.Popen(["ollama", "serve"])
time.sleep(5)

!pip install ollama --quiet
import ollama

!ollama pull llama3.1:8b
print("Servidor de Ollama listo.")


Ollama listo.


#### **INSTALAR Y LEVANTAR LLAMA STACK**

In [4]:
# Instalar dependencias

!pip install "llama-stack[starter]" llama-stack-client --quiet
!pip install "mcp<2.0" --quiet

In [5]:
# Instalar Llama Stack (servidor + cliente) y levantarlo con Ollama registrado

import subprocess
import time
import os
import yaml
import llama_stack

os.environ["OLLAMA_URL"] = "http://localhost:11434/v1"

# Generar un config de Llama Stack con el modelo de Ollama ya registrado
config_path = os.path.join(os.path.dirname(llama_stack.__file__), "distributions", "starter", "config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

config["registered_resources"]["models"].append({
    "model_id": "llama3.1:8b",
    "provider_id": "ollama",
    "provider_model_id": "llama3.1:8b",
    "model_type": "llm",
})

mi_config_path = "/content/mi_config_llama_stack.yaml"
with open(mi_config_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

# Levantar el servidor con ese config
proceso_llama_stack = subprocess.Popen(
    ["llama", "stack", "run", mi_config_path, "--port", "8321"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
time.sleep(20)  # tarda mas en levantar que Ollama

print("Proceso vivo:", proceso_llama_stack.poll() is None)
if proceso_llama_stack.poll() is not None:
    print("--- Salida del proceso ---")
    print(proceso_llama_stack.stdout.read())
else:
    from llama_stack_client import LlamaStackClient
    client = LlamaStackClient(base_url="http://localhost:8321")
    print("Llama Stack listo.")

Proceso vivo: True
Llama Stack listo.


### **CONSTRUCCIÓN DE LA LÓGICA DEL ASISTENTE**

In [6]:
# Configurar el servidor del webhook

from fastapi import FastAPI, Request, BackgroundTasks
import requests
import ollama

app = FastAPI()

mensajes_recibidos = []
usuarios_conocidos = set()

In [7]:
# Definir función de normalización de número

def normalizar_numero_mx(numero):
    if numero.startswith("521") and len(numero) == 13:
        return "52" + numero[3:]
    return numero

#### **CONSTRUIR LA MEMORIA DE CONVERSACIÓN**

Se define una estructura en memoria que guarda, por cada número de teléfono, el historial reciente de mensajes — sin esto, Llama trataría cada mensaje como si fuera el primero de la conversación. Esta es una implementación manual y simplificada del concepto; la lectura del Tema menciona **Llama Stack** como el framework que estandariza esta pieza (junto con herramientas y seguridad) para no reconstruirla en cada proyecto. **Nota:** el patrón exacto de integración de Llama Stack debe confirmarse contra su documentación vigente antes de grabar o dar la sesión en vivo.

In [8]:
# Una conversacion de Llama Stack por numero de telefono

conversaciones_por_numero = {}

def obtener_conversacion(numero):
    if numero not in conversaciones_por_numero:
        conversacion = client.conversations.create()
        conversaciones_por_numero[numero] = conversacion.id
    return conversaciones_por_numero[numero]

#### **GENERAR LA RESPUESTA CON HISTORIAL Y UNA REGLA FIJA**

Se define la lógica que arma cada respuesta: si el usuario pide hablar con una persona, se responde con un mensaje fijo de escalamiento (una decisión que conviene fijar por regla, no dejarla al modelo); en cualquier otro caso, se reenvía todo el historial reciente a Llama, para que pueda resolver referencias a mensajes anteriores.

In [9]:
# Definir la funcion de generacion de respuesta con memoria

FRASES_ESCALAMIENTO = ["hablar con alguien", "hablar con una persona", "agente humano"]

SYSTEM_PROMPT = (
    "Eres un asistente muy amable de agendado de citas de una clinica veterinaria. "
    "Responde de forma breve (2-3 lineas)."
)

def generar_respuesta(numero, texto_usuario):
    texto_low = texto_usuario.lower()
    if any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
        return "Claro, en un momento un miembro del equipo te contacta directamente."

    conversation_id = obtener_conversacion(numero)
    respuesta = client.responses.create(
        model="ollama/llama3.1:8b",
        input=texto_usuario,
        conversation=conversation_id,
        instructions=SYSTEM_PROMPT,
    )
    return respuesta.output_text

## **DESPLIEGUE DEL SERVIDOR**

### **DEFINIR LOS ENDPOINTS DEL WEBHOOK**

Se definen los dos endpoints que WhatsApp necesita: uno para la verificación inicial del webhook, y otro para recibir cada mensaje entrante. La recepción delega el trabajo pesado a `BackgroundTasks`, para responderle a Meta de inmediato sin esperar a que Llama genere la respuesta — evitando así que Meta reintente la entrega y se dupliquen las respuestas.

In [10]:
# Definir endpoint de verificacion de credenciales

@app.get("/webhook")
async def verificar_webhook(request: Request):
    params = request.query_params

    if params.get("hub.verify_token") == VERIFY_TOKEN:
        return int(params.get("hub.challenge"))

    return {"error": "token invalido"}

In [11]:
# Definir endpoint de recepcion de mensajes

@app.post("/webhook")
async def recibir_mensaje(request: Request, background_tasks: BackgroundTasks):
    body = await request.json()
    mensajes_recibidos.append(body)

    try:
        mensaje = body["entry"][0]["changes"][0]["value"]["messages"][0]
        numero = normalizar_numero_mx(mensaje["from"])
        texto = mensaje["text"]["body"]
        background_tasks.add_task(procesar_y_responder, numero, texto)
    except (KeyError, IndexError):
        pass

    return {"status": "ok"}

In [12]:
# Definir funcion de envio de respuesta

respuestas_enviadas = []

def enviar_respuesta(numero, texto):
    url = f"https://graph.facebook.com/v25.0/{PHONE_ID}/messages"
    headers = {"Authorization": f"Bearer {WHATSAPP_TOKEN}", "Content-Type": "application/json"}
    payload = {
        "messaging_product": "whatsapp",
        "to": numero,
        "type": "text",
        "text": {"body": texto}
    }

    r = requests.post(url, headers=headers, json=payload)
    respuestas_enviadas.append({"numero": numero, "status": r.status_code, "texto": r.text})

In [13]:
# Definir funcion de procesamiento: regla fija de escalamiento o respuesta de Llama

errores_procesamiento = []

def procesar_y_responder(numero, texto):
    with obtener_lock(numero):
        try:
            texto_low = texto.lower()
            if any(frase in texto_low for frase in FRASES_ESCALAMIENTO):
                enviar_respuesta(numero, "Claro, en un momento un miembro del equipo te contacta directamente.")
            else:
                respuesta = generar_respuesta(numero, texto)
                enviar_respuesta(numero, respuesta)
        except Exception as e:
            errores_procesamiento.append({"numero": numero, "texto": texto, "error": repr(e)})

In [14]:
# Un lock por numero de telefono, para serializar mensajes del mismo usuario

import threading

locks_por_numero = {}  # { numero: threading.Lock() }

def obtener_lock(numero):
    if numero not in locks_por_numero:
        locks_por_numero[numero] = threading.Lock()
    return locks_por_numero[numero]

### **EXPONER EL SERVIDOR CON NGROK**

Se abre un túnel público con ngrok y se levanta el servidor en un hilo aparte, para que Meta pueda entregarle los mensajes reales al webhook. La URL impresa (+ `/webhook`) es la que se configura como *Callback URL* en Meta for Developers (WhatsApp > Configuración de la API > Configurar webhooks).

In [15]:
!pip install pyngrok --quiet
import time
import uvicorn
import threading
from pyngrok import ngrok
#from google.colab import userdata
# 1. Configurar el Token de autenticación
ngrok.set_auth_token(userdata.get('NGROK_AUTHTOKEN'))
# 2. Desconectar y limpiar de forma estricta los túneles de la API local
try:
    tunnels = ngrok.get_tunnels()
    for tunnel in tunnels:
        ngrok.disconnect(tunnel.public_url)
    ngrok.kill()
    time.sleep(2) # Pausa estratégica para que ngrok libere el puerto y el subdominio
except Exception:
    pass

# 3. Crear el nuevo túnel (Si tienes un subdominio fijo asignado en ngrok, agrégalo aquí)
# Ejemplo si usas subdominio fijo: ngrok.connect(8000, domain="tu-subdominio.ngrok-free.dev")
tunel_publico = ngrok.connect(8000)

print("\n🚀 CONFIGURACIÓN EXITOSA 🚀")
print("==================================================================")
print("URL a configurar como Callback URL en el dashboard de Meta:")
print(f"{tunel_publico.public_url}/webhook")
print("==================================================================\n")

# 4. Levantar FastAPI/Uvicorn en un hilo secundario
def correr_servidor():
    # Asegúrate de tener definida tu variable 'app' antes de esta celda
    uvicorn.run(app, host="127.0.0.1", port=8000, log_level="info")

hilo = threading.Thread(target=correr_servidor, daemon=True)
hilo.start()
time.sleep(2)  # Dar tiempo para asegurar el arranque

INFO:     Started server process [22985]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)



🚀 CONFIGURACIÓN EXITOSA 🚀
URL a configurar como Callback URL en el dashboard de Meta:
https://mutate-stylus-facedown.ngrok-free.dev/webhook



### **PROBAR EL FLUJO COMPLETO**

Se manda un hola via colab para iniciar el flujo. esperar unos 4 minutos a que responda a mi wats. Ademas, con esto verificamos que el bot esta listo y en linea

In [16]:
# Mandar el primer mensaje para iniciar el flujo con wats y verificar que este online.

payload_1 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "5215614857733", "text": {"body": "Hola"}}
    ]}}]}]
}

r1 = requests.post("http://127.0.0.1:8000/webhook", json=payload_1)
print("Mensaje 1:", r1.status_code, r1.text)

INFO:     127.0.0.1:57468 - "POST /webhook HTTP/1.1" 200 OK
Mensaje 1: 200 {"status":"ok"}


In [17]:
_ = '''
# Mandar un segundo mensaje que depende del primero, sin repetir el servicio

payload_2 = {
    "entry": [{"changes": [{"value": {"messages": [
        {"from": "5215614857733", "text": {"body": "¿Tienen horario el sábado?"}}
    ]}}]}]
}

r2 = requests.post("http://127.0.0.1:8000/webhook", json=payload_2)
print("Mensaje 2:", r2.status_code, r2.text)

# la respuesta demora aprox. de 1 a 2 min. al ejecutar local en colab
'''

In [18]:
_ = '''
# Verificar respuestas

respuestas_enviadas
'''

In [19]:
_ = '''
# Ver mensajes recibidos

mensajes_recibidos
'''

In [20]:
_ = '''
# Verificar almacenamiento en Llama Stack

numero = normalizar_numero_mx("5215614857733")  # tu numero real de WhatsApp
conv_id = conversaciones_por_numero[numero]

items = client.conversations.items.list(conversation_id=conv_id)
for item in items.data:
    print(item.role, "->", item.content)
    '''

In [21]:
_ = '''
# Finalizar proceso de tunelización

ngrok.kill()
print('Proceso de ngrok cerrado.')
'''